In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
import numpy as np

In [2]:
# Charger les données
df = pd.read_csv("options_KR.csv")

# Conversion de la colonne 'Date' en informations numériques
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

# Encodage de la colonne 'Day_of_a_week'
le = LabelEncoder()
df['Day_of_a_week'] = le.fit_transform(df['Day_of_a_week'])


df['KOSPI200_MA5'] = df['KOSPI200'].rolling(window=5).mean()
df['KOSPI200_MA10'] = df['KOSPI200'].rolling(window=10).mean()
df['KOSPI200_Volatility_5'] = df['KOSPI200'].rolling(window=5).std()
df['KOSPI200_Volatility_10'] = df['KOSPI200'].rolling(window=10).std()
df['Call_Put_Ratio'] = df['For_Call_Netbuying_Quantity'] / (df['For_Put_Netbuying_Quantity'] + 1)
df['Call_Put_Interaction'] = df['For_Call_Netbuying_Quantity'] * df['For_Put_Netbuying_Quantity']
df['KOSPI200_Return'] = df['KOSPI200'].pct_change()
df['MA5_MA10_Diff'] = df['KOSPI200_MA5'] - df['KOSPI200_MA10']

# Nouvelles variables basées sur les corrélations
df['Volatility_Diff'] = df['KOSPI200_Volatility_5'] - df['KOSPI200_Volatility_10']
df['Momentum'] = df['KOSPI200_Return'] * df['KOSPI200_MA5']
df['Interaction_Ratio'] = df['Call_Put_Interaction'] / (df['Call_Put_Ratio'] + 1)

# Nouvelles variables pour capturer les relations inverses
df['Inverse_Call_Put_Ratio'] = 1 / (df['Call_Put_Ratio'] + 1)
df['KOSPI200_Drawdown'] = df['KOSPI200'] / df['KOSPI200'].cummax() - 1
df['Inverse_Momentum'] = -df['KOSPI200_Return'] * df['KOSPI200_MA5']
df['Volatility_Ratio'] = df['KOSPI200_Volatility_5'] / (df['KOSPI200_Volatility_10'] + 1e-6)
df['Cum_Call_Put_Diff'] = df['For_Call_Netbuying_Quantity'].cumsum() - df['For_Put_Netbuying_Quantity'].cumsum()


# Ratio dynamique entre volatilité et rendement
df['Volatility_Return_Ratio'] = df['KOSPI200_Volatility_5'] / (df['KOSPI200_Return'].abs() + 1e-6)


# Trier les données par date
df_sorted = df.sort_values(by='Date').dropna()

# Séparer les 20% des données les plus récentes pour le test
split_index = int(len(df_sorted) * 0.8)
df_train = df_sorted.iloc[:split_index]
df_test = df_sorted.iloc[split_index:]

# Date,VKOSPI,KOSPI200,Open_interest,For_KOSPI_Netbuying_Amount,For_Future_Netbuying_Quantity,For_Call_Netbuying_Quantity,For_Put_Netbuying_Quantity,Indiv_Future_Netbuying_Quantity,Indiv_Call_Netbuying_Quantity,Indiv_Put_Netbuying_Quantity,PCRatio,Day_till_expiration,Day_of_a_week
# Définir les variables explicatives (X) et la cible (y)
X_train = df_train.drop(columns=['VKOSPI', 'Date', 'Month', 'Year', 'Day', 'Day_of_a_week', 'PCRatio', 'Day_till_expiration'])
y_train = df_train['VKOSPI']
X_test = df_test.drop(columns=['VKOSPI', 'Date', 'Month', 'Year', 'Day', 'Day_of_a_week', 'PCRatio', 'Day_till_expiration'])
y_test = df_test['VKOSPI']

# Standardisation des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modèle Gradient Boosting optimisé
gb_model = GradientBoostingRegressor(
    n_estimators=1500,
    learning_rate=0.01,
    max_depth=3,
    min_samples_leaf=2,
    min_samples_split= 10
)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

# Calcul de la RMSE et du R²
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)
print("RMSE sur les 20% des données les plus récentes avec Gradient Boosting:", rmse_gb)
print("Coefficient de détermination R²:", r2_gb)

# Vérifier si la RMSE est inférieure à 1.5
if rmse_gb < 1.5:
    print("✔️ La performance est satisfaisante : RMSE < 1.5")
else:
    print("❌ La performance est insuffisante : RMSE >= 1.5")


RMSE sur les 20% des données les plus récentes avec Gradient Boosting: 2.152342220560766
Coefficient de détermination R²: 0.2565194090490127
❌ La performance est insuffisante : RMSE >= 1.5
